In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
create widget text storageName default "adlsproyecto2026";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS ventas CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS ventas
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS ventas.raw;
DROP SCHEMA IF EXISTS ventas.bronze;
DROP SCHEMA IF EXISTS ventas.silver;
DROP SCHEMA IF EXISTS ventas.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS ventas.raw;
CREATE SCHEMA IF NOT EXISTS ventas.bronze;
CREATE SCHEMA IF NOT EXISTS ventas.silver;
CREATE SCHEMA IF NOT EXISTS ventas.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS ventas.bronze.sales (
sale_id integer,
sale_year integer,
store_id integer,
productName string,
sale_amount decimal(10,2),
sale_quantity integer,
ingestion_date timestamp
)
USING DELTA
PARTITIONED BY (sale_year)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sales"

In [0]:
CREATE TABLE IF NOT EXISTS ventas.bronze.stores (
store_id integer,
store_ref string,
store_name string,
location string,
country string,
store_type string,
floor_area_sqm integer,
employee_count integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/stores"

In [0]:
CREATE TABLE IF NOT EXISTS ventas.bronze.brands (
  brand_id integer,
  brand_ref string,
  name string,
  nationality string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/brands"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS ventas.silver.sales_stores_transformed (
-- Campos de Ventas (provenientes de tu tabla races adaptada)
  sale_id integer,
  sale_year integer,
  store_id integer,
  product_name string,
  sale_amount decimal(10,2),
  sale_quantity integer,
    
  -- Campos de Tiendas (provenientes de tu tabla stores)
  store_ref string,
  store_name string,   -- Reemplaza a 'name' para no confundir con productName
  location string,
  country string,
  store_type string,
  floor_area_sqm integer,
  employee_count integer,
  
  -- Campos Transformados (Adaptación 1 a 1 de tus campos de F1)
  area_category string,       -- Reemplaza a 'altitude_category' (Ej: 'Tienda Grande', 'Tienda Pequeña')
  years_diferences integer,   -- Se mantiene igual (Ej: Año actual - sale_year)
  area_per_employee integer,   -- Reemplaza a 'lat_diff' (KPI: floor_area_sqm / employee_count)
  sale_type string,           -- Reemplaza a 'race_type' (Ej: 'Mayorista' si sale_quantity > X, sino 'Minorista')
  high_efficiency string,      -- Reemplaza a 'near_equator' (Ej: 'Si' si vende mucho con pocos empleados)

  transformation_timestamp timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/sales_stores_transformed"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS ventas.golden.golden_sales_partitioned (
  sale_year integer,
  product_name string,
  
  -- Métricas Agregadas
  cantidad_productos_vendidos long,  -- Se mantiene (Cantidad total de ventas)
  max_sale_amount decimal(10,2),     -- Reemplaza a 'max_altitude' (Monto de venta máximo del año)
  min_sale_amount decimal(10,2),     -- Reemplaza a 'min_altitude' (Monto de venta mínimo del año)
  
  -- Dimensiones de agrupación
  country string,
  sale_type string,           -- Reemplaza a 'race_type'
  high_efficiency string      -- Reemplaza a 'near_equator'  
)
USING DELTA
PARTITIONED BY (sale_year,product_name)
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_sales_partitioned";